In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np
np.set_printoptions(legacy="1.13")

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('Attribute_List.xlsx') in f:

                    file_list.append(f)
file_list

In [4]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [5]:
len(file_link)

9

In [ ]:
file_link[1]

In [7]:
cols=['Brand']
df_s = pd.DataFrame(columns=cols)

In [ ]:
for i in range(len(file_link)):
    file=file_link[i]
    print(file)
    workbook=openpyxl.load_workbook(file)
    sheetname=workbook.sheetnames
    print(sheetname)
    for s in sheetname:
        if (("FBG_Attributes" not in s)):
            print(s)
            df=pd.read_excel(file_link[i],sheet_name=s, keep_default_na=False, na_values=[''])
            df['Source']=file_link[i].split('\\')[-1].split('.')[0]  
            df_s=pd.concat([df_s,df])

In [9]:
df_s['PAName']=df_s['PAName'].str.strip()

In [11]:
df_AC=pd.read_excel(fr"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Autocare\20250731_Autocare_PCAdb.xlsx")
df_AC['Attribute_Type']="Autocare_Attribute"

In [ ]:
df_AC

In [ ]:
df_s.Brand.unique()

In [14]:
df_PNS=df_s[['PartNumber','PartTerminologyName','Brand']].drop_duplicates(ignore_index=True)

In [ ]:
df_merged=df_PNS.merge(df_AC,how='left')
df_merged

In [ ]:
df_merged

In [17]:
df_s["Key"]=df_s['PartNumber'].astype(str)+df_s['PartTerminologyName'].astype(str)+df_s['PAName'].astype(str)

In [18]:
df_merged["Key"]=df_merged['PartNumber'].astype(str)+df_merged['PartTerminologyName'].astype(str)+df_merged['PAName'].astype(str)

In [ ]:
df_s.columns

In [ ]:
df_AC

In [21]:
df_Final=df_merged.merge(df_s,how="outer")

In [ ]:
df_Final

In [23]:
df_Final['AC_AttributeCount']=np.where(df_Final["Attribute_Type"].notna(), 1 , 0)
df_Final['Attribute_Value']=np.where((df_Final["Attribute_Type"].notna())  & (df_Final["Value"].notna()), 1 , 0)
# df_Final=df_Final.dropna(subset='Autocare')
df_Final['Attribute_Type']=np.where((df_Final["Attribute_Type"].notna())  , df_Final["Attribute_Type"] , 'FBG_Attribute')

In [ ]:
df_Final

In [ ]:
df_BM=pd.read_excel(fr"BrandMap.xlsx",sheet_name="BrandMap")
df_BM

In [ ]:
df_BM.columns

In [ ]:
df_mapped=df_Final.merge(df_BM,how="outer",left_on=["Brand"], right_on=["BrandName"],suffixes=('_match', '_filter'))
df_mapped

In [ ]:
df_mapped.columns

In [29]:
df_final_List=df_mapped[['Product Group','Source_filter','BrandName','Brand_filter', 'PartNumber', 'PartTerminologyName', 'Status', 'PAName', 'Attribute_Type', 'Value', 'AC_AttributeCount', 'Attribute_Value','Key','Parent-Child- Customer Brand']]

In [30]:
df_final_List.rename(columns={'Brand_filter': 'Brand','Source_filter':'Source'}, inplace=True)

In [ ]:
df_final_List.columns

In [ ]:
df_final_List=df_final_List.drop_duplicates()
df_final_List = df_final_List[df_final_List['PartTerminologyName'].notna()]
df_final_List = df_final_List[df_final_List['Product Group'].notna()]
df_final_List

In [33]:
df_final_List['Product Group'].unique()

array(['Repair', 'Brakes', 'Towing', 'Steering/ Electronics', 'Filter',
       'Lighting', 'Wipers'], dtype=object)

In [ ]:
# chunk_size=1000000
# # Create a list of DataFrames by splitting the original DataFrame
# df_chunks = [df_mapped.iloc[i:i + chunk_size] for i in range(0, len(df_mapped), chunk_size)]

# with pd.ExcelWriter(r"path/filename.xlsx") as writer:  # doctest: +SKIP
#     for i, chunk in enumerate(df_chunks):
#         sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
#         chunk.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
df_final_List.to_csv(r"PAth/filename.csv",index=False)

In [36]:
cols=['Brand']
df_a = pd.DataFrame(columns=cols)

In [ ]:
file_link[1].split('\\')[-1].split('.')[0]  

In [ ]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_Match.iloc[i:i + chunk_size] for i in range(0, len(df_Match), chunk_size)]


with pd.ExcelWriter(OFolder+'\\'+'Horizon_Global_Attribute_List.xlsx') as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
for i in range(len(file_link)):
    file=file_link[i]
    workbook=openpyxl.load_workbook(file)
    sheetname=workbook.sheetnames
    print(sheetname)
    for s in sheetname:
        if (("FBG_Attributes" in s)):
            print(s)
            df=pd.read_excel(file_link[i],sheet_name=s)
            df['Source']=file_link[i].split('\\')[-1].split('.')[0]  
            df_a=pd.concat([df_a,df])

In [ ]:
df_a

In [ ]:
df_a.to_csv(r"Path/Filename.csv",index=False)